In [26]:
import numpy as np

#Activator类实现激活函数，其中forward方法实现了前向计算，backward方法则是计算导数
class ReluActivator(object):
    def forward(self,weighted_input):
        return max(0,weighted_input)
    def backward(self,output):  # RELU的导函数
        return 1 if output > 0 else 0

class IdentityActivator(object):
    def forward(self, weighted_input):
        return weighted_input
    def backward(self, output):
        return 1

def element_wise_op(array,op):
    """ 对array逐元素应用op操作（激活函数）"""
    for i in np.nditer(array, op_flags=['readwrite']): # 使用 np.nditer 遍历数组，允许读写元素
        # 将当前位置的元素值替换为 op(i) 的结果
        i[...] = op(i)

In [27]:
# 用RecurrentLayer类来实现一个循环层。下面的代码是初始化一个循环层，可以在构造函数中设置卷积层的超参数。循环层有两个权重数组，U和W。
class RecurrentLayer(object):
    def __init__(self, input_width, state_width, activator, learning_rate):
        self.input_width = input_width
        self.state_width = state_width
        self.activator = activator
        self.learning_rate = learning_rate
        self.times = 0
        self.state_list = []      # 保存各时刻的激活值 h(t)
        self.net_list = []        # 保存各时刻的加权输入 net(t) = U*x(t) + W*h(t-1)
        self.input_list = []      # 保存各时刻的输入 x(t)
        self.state_list.append(np.zeros((state_width, 1)))
        self.net_list.append(np.zeros((state_width, 1)))  # 占位，t=0无意义
        # 初始化权重
        self.U = np.random.uniform(-1e-4, 1e-4, (state_width, input_width))
        self.W = np.random.uniform(-1e-4, 1e-4, (state_width, state_width))

    def forward(self, input_array):
        self.times += 1
        # 保存当前输入
        self.input_list.append(input_array)
        # 计算净输入 net = U*x + W*h_prev
        net = np.dot(self.U, input_array) + np.dot(self.W, self.state_list[-1])
        self.net_list.append(net)
        # 激活得到状态
        state = net.copy()
        element_wise_op(state, self.activator.forward)
        self.state_list.append(state)

    def backward(self, sensitivity_array, activator):
        """
        sensitivity_array: 损失对最后一个时刻输出（状态）的导数，形状 (state_width,1)
        activator: 用于计算激活函数导数的激活器（应与前向相同）
        """
        self.calc_delta(sensitivity_array, activator)
        self.calc_gradient()

    def calc_delta(self, sensitivity_array, activator):
        # delta_list: 误差项，长度 = times+1, delta_list[t] 对应时刻 t 的误差项（对 net(t) 的偏导）
        self.delta_list = [np.zeros((self.state_width, 1)) for _ in range(self.times + 1)]
        # 最后一个时刻的 delta 由外部传入的 sensitivity_array 给出（假设损失直接对 h(T) 求导）
        # 注意：delta = dL/dnet = dL/dh * dh/dnet = sensitivity_array * f'(net)
        # 因此需要乘以激活函数的导数
        net_T = self.net_list[-1]
        f_prime = np.zeros_like(net_T)
        # 逐元素计算导数
        for i in range(self.state_width):
            f_prime[i,0] = activator.backward(net_T[i,0])
        self.delta_list[-1] = sensitivity_array * f_prime   # 逐元素乘

        # 反向递推 delta(t) = (W^T * delta(t+1)) * f'(net(t))
        for k in range(self.times - 1, 0, -1):
            self.calc_delta_k(k, activator)

    def calc_delta_k(self, k, activator):
        """根据 k+1 时刻的 delta 计算 k 时刻的 delta"""
        # 获取当前时刻的净输入 net(k)
        net_k = self.net_list[k]
        # 计算 f'(net(k))
        f_prime = np.zeros_like(net_k)
        for i in range(self.state_width):
            f_prime[i,0] = activator.backward(net_k[i,0])
        # delta(k) = (W^T * delta(k+1)) * f'(net(k))
        delta_next = self.delta_list[k+1]                     # 形状 (state_width,1)
        w_transpose_delta = np.dot(self.W.T, delta_next)      # (state_width,1)
        self.delta_list[k] = w_transpose_delta * f_prime      # 逐元素乘

    def calc_gradient(self):
        # 初始化梯度累加器
        self.W_grad = np.zeros_like(self.W)   # 对 W 的梯度
        self.U_grad = np.zeros_like(self.U)   # 对 U 的梯度

        # 遍历每个时刻 t = 1..times
        for t in range(1, self.times + 1):
            # 对 W 的梯度：delta(t) * h(t-1)^T
            self.W_grad += np.dot(self.delta_list[t], self.state_list[t-1].T)
            # 对 U 的梯度：delta(t) * x(t)^T
            x_t = self.input_list[t-1]   # input_list 索引 0 对应第1个输入
            self.U_grad += np.dot(self.delta_list[t], x_t.T)

    def update(self):
        """梯度下降更新权重"""
        self.W -= self.learning_rate * self.W_grad
        self.U -= self.learning_rate * self.U_grad

    def reset_state(self):
        self.times = 0
        self.state_list = [np.zeros((self.state_width, 1))]
        self.net_list = [np.zeros((self.state_width, 1))]
        self.input_list = []

In [28]:
def data_set():
    x = [np.array([[1], [2], [3]]),np.array([[2], [3], [4]])]
    d = np.array([[1], [2]])
    return x, d

In [29]:
def gradient_check():
    """梯度检查"""
    error_function = lambda o: o.sum()
    rl = RecurrentLayer(3, 2, IdentityActivator(), 1e-3)

    x, d = data_set()
    rl.forward(x[0])
    rl.forward(x[1])

    # 假设损失对最后一个时刻的状态的梯度为全1
    sensitivity_array = np.ones(rl.state_list[-1].shape, dtype=np.float64)
    rl.backward(sensitivity_array, IdentityActivator())

    epsilon = 1e-4
    for i in range(rl.W.shape[0]):
        for j in range(rl.W.shape[1]):
            # 数值梯度
            rl.W[i, j] += epsilon
            rl.reset_state()
            rl.forward(x[0])
            rl.forward(x[1])
            err1 = error_function(rl.state_list[-1])

            rl.W[i, j] -= 2 * epsilon
            rl.reset_state()
            rl.forward(x[0])
            rl.forward(x[1])
            err2 = error_function(rl.state_list[-1])

            expect_grad = (err1 - err2) / (2 * epsilon)
            rl.W[i, j] += epsilon   # 恢复原值

            # 重新计算解析梯度（因为W被修改后又恢复了，但之前的backward结果已过期）
            rl.reset_state()
            rl.forward(x[0])
            rl.forward(x[1])
            rl.backward(sensitivity_array, IdentityActivator())
            actual_grad = rl.W_grad[i, j]

            print(f'weights({i},{j}): expected - actual = {expect_grad:.6f} - {actual_grad:.6f}')

In [30]:
def test():
    l = RecurrentLayer(3, 2, ReluActivator(), 1e-3)
    x, d = data_set()
    l.forward(x[0])
    l.forward(x[1])
    l.backward(d, ReluActivator())
    return l

In [31]:
test()

In [32]:
gradient_check()

weights(0,0): expected - actual = 0.000303 - 0.000303
weights(0,1): expected - actual = 0.000156 - 0.000156
weights(1,0): expected - actual = 0.000303 - 0.000303
weights(1,1): expected - actual = 0.000156 - 0.000156
